# Kimi K3: training from a complete profile

Each profile groups `data.yaml`, `model.yaml` and `training.yaml`. The public pipeline keeps three explicit steps: data, model and training. The final cell starts a real training run, so review the selected profile first.

In [ ]:
from pathlib import Path

from configuration import resolve_kimi_pipeline_profile
from data import build_dataloaders_from_yaml
from src import build_model_from_yaml
from training import train_kimi_from_yaml

In [ ]:
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()

# Change only PROFILE_NAME to switch the complete data/model/training recipe.
PROFILE_NAME = 't4_retrieval'
# PROFILE_NAME = 'cpu_smoke'
# PROFILE_NAME = 'low_gpu'
# PROFILE_NAME = 't4_wikitext'
# PROFILE_NAME = 'gpu_24gb'
# PROFILE_NAME = 'gpu_48gb'
# PROFILE_NAME = 'gpu_80gb'
# PROFILE_NAME = 'canonical'  # Metadata validation only; do not instantiate casually.

PROFILE = resolve_kimi_pipeline_profile(
    ROOT / 'config/kimi_full_pipeline' / PROFILE_NAME
)
DATA_YAML = PROFILE.data
MODEL_YAML = PROFILE.model
TRAINING_YAML = PROFILE.training

## 1. Data

Build the tokenizer, datasets, loaders and the loader factory required by Progressive Context Curriculum. Hugging Face profiles may download or tokenize data here; `t4_retrieval` is fully synthetic and download-free.

In [ ]:
data_bundle = build_dataloaders_from_yaml(DATA_YAML)
data_bundle.config

## 2. Model

Passing the data bundle validates vocabulary compatibility and resolves tokenizer special-token IDs before model construction.

In [ ]:
model = build_model_from_yaml(MODEL_YAML, data_bundle=data_bundle)
model.config

## 3. Training

The next cell delegates exactly once to the master `train_kimiK3` orchestrator. **Running it starts real training.**

In [ ]:
result = train_kimi_from_yaml(
    TRAINING_YAML,
    model=model,
    data=data_bundle,
)
result['last_checkpoint']